# Advanced Prompt Chaining: Job Application Analyzer

This notebook demonstrates a **4-step linear prompt chain** that evaluates a job candidate by processing a job description and a resume through a series of LLM calls.

Each step produces a **structured Pydantic output** that is passed as context to the next step.

## 1. Setup

In [1]:
from dotenv import load_dotenv
from typing import List
from pydantic import BaseModel, Field

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.1)

## 2. Pydantic Output Models

Each intermediate step in the chain returns a validated Pydantic object instead of raw text. This prevents format errors and makes it easy to access specific fields in downstream steps.

In [2]:
class JobRequirements(BaseModel):
    """Structured requirements extracted from a job description."""
    job_title: str = Field(description="The job title")
    required_skills: List[str] = Field(description="List of required technical skills")
    nice_to_have_skills: List[str] = Field(description="List of optional/bonus skills")
    years_of_experience: int = Field(description="Minimum years of experience required")
    seniority_level: str = Field(description="e.g. Junior, Mid-level, Senior, Lead")


class CandidateScore(BaseModel):
    """Evaluation of a candidate's resume against the job requirements."""
    overall_score: int = Field(description="Overall match score from 0 to 10")
    matched_skills: List[str] = Field(description="Skills the candidate has that match requirements")
    missing_skills: List[str] = Field(description="Required skills that are absent from the resume")
    experience_meets_requirement: bool = Field(description="True if candidate's experience meets the minimum")
    summary: str = Field(description="One-sentence summary of the candidate's fit")


class InterviewPlan(BaseModel):
    """A structured interview plan tailored to the candidate's gaps."""
    technical_questions: List[str] = Field(description="3-4 technical questions targeting skill gaps")
    behavioral_questions: List[str] = Field(description="2-3 behavioral questions")
    recommended_interview_format: str = Field(description="e.g. 'Take-home + 1 technical round', '2 panel interviews'")

## 3. Build the 4-Step Chain

Each step uses `llm.with_structured_output(Model)` to bind the Pydantic schema to the LLM call, so the output is automatically validated and parsed. The final step uses `StrOutputParser` to produce a free-form narrative report.

In [3]:
# ── Step 1: Extract structured requirements from the job description ─────────
prompt_step1 = ChatPromptTemplate.from_messages([
    ("system", "You are an expert technical recruiter. Extract structured information from job postings."),
    ("human", "Extract the requirements from this job description:\n\n{job_description}"),
])

chain_step1 = prompt_step1 | llm.with_structured_output(JobRequirements)


# ── Step 2: Score the candidate's resume against the requirements ────────────
prompt_step2 = ChatPromptTemplate.from_messages([
    ("system", "You are a senior hiring manager. Evaluate candidates objectively and fairly."),
    ("human", (
        "Evaluate this candidate's resume against the job requirements below.\n\n"
        "Job Title: {job_title}\n"
        "Required Skills: {required_skills}\n"
        "Nice-to-have Skills: {nice_to_have_skills}\n"
        "Minimum Years of Experience: {years_of_experience}\n"
        "Seniority Level: {seniority_level}\n\n"
        "Candidate Resume:\n{resume}"
    )),
])

chain_step2 = prompt_step2 | llm.with_structured_output(CandidateScore)


# ── Step 3: Build an interview plan targeting the candidate's gaps ────────────
prompt_step3 = ChatPromptTemplate.from_messages([
    ("system", "You are an experienced technical interviewer. Design targeted interview plans."),
    ("human", (
        "Create an interview plan for a candidate applying for {job_title}.\n\n"
        "Overall Score: {overall_score}/10\n"
        "Matched Skills: {matched_skills}\n"
        "Missing Skills: {missing_skills}\n"
        "Experience Meets Requirement: {experience_meets_requirement}\n\n"
        "Focus technical questions on the missing skills and experience gaps."
    )),
])

chain_step3 = prompt_step3 | llm.with_structured_output(InterviewPlan)


# ── Step 4: Write a hiring recommendation report ─────────────────────────────
prompt_step4 = ChatPromptTemplate.from_messages([
    ("system", "You are a hiring manager writing an internal report. Be concise, direct, and professional."),
    ("human", (
        "Write a hiring recommendation report for the following candidate.\n\n"
        "Position: {job_title}\n"
        "Score: {overall_score}/10\n"
        "Fit Summary: {summary}\n"
        "Missing Skills: {missing_skills}\n"
        "Recommended Interview Format: {recommended_interview_format}\n"
        "Technical Questions to Ask: {technical_questions}\n\n"
        "The report should include: a recommendation (Advance / Hold / Reject), "
        "a brief justification, and next steps."
    )),
])

chain_step4 = prompt_step4 | llm | StrOutputParser()

## 4. Input Data

In [4]:
job_description = """
We are looking for a Senior Machine Learning Engineer to join our AI Platform team.

Responsibilities:
- Design and deploy production ML models and pipelines at scale.
- Collaborate with data scientists to move experiments into production.
- Build and maintain MLOps infrastructure.

Requirements:
- 5+ years of experience in software engineering or data science.
- Strong proficiency in Python and ML frameworks (PyTorch or TensorFlow).
- Experience with cloud platforms (AWS or GCP).
- Solid understanding of MLOps practices: CI/CD for ML, model monitoring, feature stores.
- Experience with containerization (Docker, Kubernetes).

Nice to have:
- Experience with LLMs and LangChain.
- Familiarity with Apache Spark or Flink for large-scale data processing.
- Open-source contributions.
"""

resume = """
Jane Doe | jane.doe@email.com | github.com/janedoe

EXPERIENCE
ML Engineer — DataCorp (3 years)
  - Built and deployed recommendation models using PyTorch in production.
  - Created data pipelines with Python and Airflow.
  - Some experience with AWS S3 and EC2.

Junior Data Scientist — StartupXYZ (1.5 years)
  - Developed classification models in scikit-learn.
  - Wrote ETL scripts in Python.

SKILLS
Python, PyTorch, scikit-learn, SQL, Airflow, Git, Docker (basic), AWS (S3, EC2)

EDUCATION
B.Sc. Computer Science — State University
"""

## 5. Run the Pipeline (step-by-step)

Each step's output object is unpacked into a plain dict and passed to the next prompt template. This is the key mechanic of linear prompt chaining with structured outputs.

In [5]:
# ── Step 1 ───────────────────────────────────────────────────────────────────
print("Step 1: Extracting job requirements...")
requirements: JobRequirements = chain_step1.invoke({
    "job_description": job_description,
})

# ── Step 2 ───────────────────────────────────────────────────────────────────
print("Step 2: Scoring the candidate...")
score: CandidateScore = chain_step2.invoke({
    **requirements.model_dump(),
    "resume": resume,
})

# ── Step 3 ───────────────────────────────────────────────────────────────────
print("Step 3: Building interview plan...")
interview: InterviewPlan = chain_step3.invoke({
    "job_title": requirements.job_title,
    **score.model_dump(),
})

# ── Step 4 ───────────────────────────────────────────────────────────────────
print("Step 4: Writing hiring recommendation...\n")
report: str = chain_step4.invoke({
    "job_title": requirements.job_title,
    "overall_score": score.overall_score,
    "summary": score.summary,
    "missing_skills": score.missing_skills,
    "recommended_interview_format": interview.recommended_interview_format,
    "technical_questions": interview.technical_questions,
})

Step 1: Extracting job requirements...
Step 2: Scoring the candidate...
Step 3: Building interview plan...
Step 4: Writing hiring recommendation...



### Inspect Intermediate Outputs

Because each step returns a Pydantic model, we can access individual fields directly — no string parsing required.

In [6]:
# Step 1 output — JobRequirements
print("=== Step 1: Job Requirements ===")
print(f"  Title         : {requirements.job_title}")
print(f"  Seniority     : {requirements.seniority_level}")
print(f"  Min. Exp.     : {requirements.years_of_experience} years")
print(f"  Required      : {requirements.required_skills}")
print(f"  Nice-to-have  : {requirements.nice_to_have_skills}")

# Step 2 output — CandidateScore
print("\n=== Step 2: Candidate Score ===")
print(f"  Score         : {score.overall_score}/10")
print(f"  Summary       : {score.summary}")
print(f"  Matched Skills: {score.matched_skills}")
print(f"  Missing Skills: {score.missing_skills}")
print(f"  Exp. OK?      : {score.experience_meets_requirement}")

# Step 3 output — InterviewPlan
print("\n=== Step 3: Interview Plan ===")
print(f"  Format        : {interview.recommended_interview_format}")
print("  Technical Questions:")
for q in interview.technical_questions:
    print(f"    • {q}")
print("  Behavioral Questions:")
for q in interview.behavioral_questions:
    print(f"    • {q}")

=== Step 1: Job Requirements ===
  Title         : Senior Machine Learning Engineer
  Seniority     : Senior
  Min. Exp.     : 5 years
  Required      : ['Python', 'PyTorch', 'TensorFlow', 'AWS', 'GCP', 'MLOps', 'CI/CD for ML', 'Model Monitoring', 'Feature Stores', 'Docker', 'Kubernetes']
  Nice-to-have  : ['LLMs', 'LangChain', 'Apache Spark', 'Flink', 'Open-source contributions']

=== Step 2: Candidate Score ===
  Score         : 4/10
  Summary       : Jane has solid experience with Python and PyTorch for model deployment but lacks the comprehensive MLOps, cloud, and specific tool experience required for a Senior Machine Learning Engineer role, and her total experience is slightly below the minimum.
  Matched Skills: ['Python', 'PyTorch', 'AWS', 'Docker']
  Missing Skills: ['TensorFlow', 'GCP', 'MLOps', 'CI/CD for ML', 'Model Monitoring', 'Feature Stores', 'Kubernetes']
  Exp. OK?      : False

=== Step 3: Interview Plan ===
  Format        : Take-home assignment (MLOps/System Design 

In [7]:
# Step 4 output — Final hiring recommendation report
print("=== Step 4: Hiring Recommendation Report ===\n")
print(report)

=== Step 4: Hiring Recommendation Report ===

**Candidate:** Jane
**Position:** Senior Machine Learning Engineer

**Recommendation:** Reject

**Justification:**
Jane demonstrates solid experience with Python and PyTorch for model deployment. However, she lacks the comprehensive MLOps, cloud (GCP), and specific tool experience (TensorFlow, Kubernetes, CI/CD for ML, Model Monitoring, Feature Stores) required for a Senior Machine Learning Engineer role. Her total experience is also slightly below the minimum for this level. The identified skill gaps are significant for the responsibilities of this senior position.

**Next Steps:**
Close application.


## 6. Unified LCEL Chain

The entire 4-step pipeline can be expressed as a **single chain object** using two LCEL primitives:

- **`RunnablePassthrough.assign(key=chain)`** — runs a sub-chain and adds its output to the current dict under `key`, keeping all other fields intact.
- **`RunnableLambda(fn)`** — applies an arbitrary Python function to transform the dict between steps (used here to unpack Pydantic models with `.model_dump()`).

In [8]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

full_pipeline = (
    # Step 1: run chain_step1 and store the result under "requirements"
    RunnablePassthrough.assign(requirements=chain_step1)

    # Unpack the Pydantic model so step 2's prompt variables are available.
    # Also carry the original "resume" field forward.
    | RunnableLambda(lambda x: {
        **x["requirements"].model_dump(),
        "resume": x["resume"],
    })

    # Step 2: run chain_step2 and store the result under "score"
    | RunnablePassthrough.assign(score=chain_step2)

    # Unpack the score model. Keep "job_title" from step 1 for later steps.
    | RunnableLambda(lambda x: {
        "job_title": x["job_title"],
        **x["score"].model_dump(),
    })

    # Step 3: run chain_step3 and store the result under "interview"
    | RunnablePassthrough.assign(interview=chain_step3)

    # Select only the fields that step 4's prompt template expects.
    | RunnableLambda(lambda x: {
        "job_title": x["job_title"],
        "overall_score": x["overall_score"],
        "summary": x["summary"],
        "missing_skills": x["missing_skills"],
        "recommended_interview_format": x["interview"].recommended_interview_format,
        "technical_questions": x["interview"].technical_questions,
    })

    # Step 4: produce the final narrative report (plain string)
    | chain_step4
)

In [9]:
final_report = full_pipeline.invoke({
    "job_description": job_description,
    "resume": resume,
})

print(final_report)

**Hiring Recommendation Report**

**Position:** Senior Machine Learning Engineer
**Candidate Score:** 4/10

**Recommendation:** Reject

**Justification:**
The candidate falls significantly short of the required experience and critical skills for a Senior Machine Learning Engineer role. Key deficiencies include MLOps, advanced cloud platforms (GCP), orchestration (Kubernetes), and specific frameworks (TensorFlow). While core ML skills are present, the absence of these senior-level capabilities makes them unsuitable for this position.

**Next Steps:**
Inform the candidate of the decision and close the application.
